# Goldset Final Grade 결정

3번의 LLM Judge 결과를 합산하여 최종 grade를 확정하고,
query_id별 grade 2~3 문서 충분성을 점검한다.

## 판정 규칙
| 조건 | 결과 | confidence |
|---|---|---|
| 완전 일치 (3/3 동일) | gold 포함 | HIGH |
| 다수결 (2/3 동일, outlier 차이 무관) | gold 포함 | MEDIUM |
| 완전 불일치 (3개 모두 다른 값) | gold 제외 | LOW |

## 데이터 소스
| Judge | 파일 |
|---|---|
| 1st | `goldset_all_scenarios1.json` |
| 2nd | `goldset_all_scenarios2.json` |
| 3rd | `goldset_all_scenarios3.json` |

## 1. 데이터 로드

In [12]:
import json
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

DATA_DIR = Path("../dataset")

def load_json(path: Path) -> List[Dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)

judge1 = load_json(DATA_DIR / "goldset_all_scenarios1.json")
judge2  = load_json(DATA_DIR / "goldset_all_scenarios2.json")
judge3 = load_json(DATA_DIR / "goldset_all_scenarios3.json")

print(f"1st judge: {len(judge1):,}건")
print(f"2nd judge: {len(judge2):,}건" )
print(f"3rd judge: {len(judge3):,}건")

1st judge: 2,543건
2nd judge: 2,543건
3rd judge: 2,543건


In [13]:
# 각 Judge 파일의 Failed(relevance_grade=None) 건수 확인
from collections import defaultdict

def failed_summary(records: list, label: str):
    total  = len(records)
    failed = [r for r in records if r.get("relevance_grade") is None]
    by_qid = defaultdict(int)
    for r in failed:
        by_qid[r["query_id"]] += 1

    print(f"=== {label}  (총 {total:,}건 / 실패 {len(failed):,}건) ===")
    if failed:
        print(f"  {'query_id':>10}  {'failed':>7}")
        print(f"  {'-'*20}")
        for qid in sorted(by_qid):
            print(f"  {qid:>10}  {by_qid[qid]:>7}")
    else:
        print("  실패 없음")
    print()

failed_summary(judge1, "1st judge  (goldset_all_scenarios1.json)")
failed_summary(judge2, "2nd judge  (goldset_all_scenarios2.json)")
failed_summary(judge3, "3rd judge  (goldset_all_scenarios3.json)")

=== 1st judge  (goldset_all_scenarios1.json)  (총 2,543건 / 실패 0건) ===
  실패 없음

=== 2nd judge  (goldset_all_scenarios2.json)  (총 2,543건 / 실패 0건) ===
  실패 없음

=== 3rd judge  (goldset_all_scenarios3.json)  (총 2,543건 / 실패 0건) ===
  실패 없음



## 2. 3개 Judge 결과 병합

In [14]:
# isbn + query_id → grade 매핑 (None 이외는 모두 int로 정규화)
def to_int_grade(v):
    return int(v) if v is not None else None

g1 = {(r["isbn"], r["query_id"]): to_int_grade(r["relevance_grade"]) for r in judge1}
g2 = {(r["isbn"], r["query_id"]): to_int_grade(r["relevance_grade"]) for r in judge2}
g3 = {(r["isbn"], r["query_id"]): to_int_grade(r["relevance_grade"]) for r in judge3}

all_keys = set(g1) | set(g2) | set(g3)
missing  = {k for k in all_keys if not (k in g1 and k in g2 and k in g3)}
print(f"전체 문서 수: {len(all_keys):,}")
print(f"3개 judge 모두 있는 문서: {len(all_keys) - len(missing):,}")
if missing:
    print(f"⚠️  일부 judge 누락: {len(missing)}건 — 해당 문서는 제외됩니다")

전체 문서 수: 2,543
3개 judge 모두 있는 문서: 2,543


## 3. 최종 Grade 결정 함수

In [15]:
def decide_final_grade(grades: List[Optional[int]]) -> Dict[str, Any]:
    # None(LLM 오류) 포함 시 판정 불가 → 제외
    if any(g is None for g in grades):
        return {"final_grade": None, "confidence": "LOW", "gold_status": "excluded"}

    counter = Counter(grades)
    majority_grade, majority_count = counter.most_common(1)[0]

    # 완전 일치 (3/3) → HIGH
    if majority_count == 3:
        return {"final_grade": majority_grade, "confidence": "HIGH",   "gold_status": "included"}

    # 다수결 (2/3 동일 값) → MEDIUM  ※ outlier와의 차이는 무관
    if majority_count == 2:
        return {"final_grade": majority_grade, "confidence": "MEDIUM", "gold_status": "included"}

    # 완전 불일치 (3개 모두 다른 값) → 제외
    return {"final_grade": None, "confidence": "LOW", "gold_status": "excluded"}

## 4. 전체 문서에 판정 적용

In [16]:
# 메타데이터는 1st judge 기준으로 사용
meta = {(r["isbn"], r["query_id"]): r for r in judge1}

records = []
for key in sorted(all_keys - missing):
    isbn, query_id = key
    grades = [g1[key], g2[key], g3[key]]
    verdict = decide_final_grade(grades)

    base = meta[key]
    records.append({
        "query_id"   : query_id,
        "isbn"       : isbn,
        "title"      : base.get("title", ""),
        "grade_j1"   : grades[0],
        "grade_j2"   : grades[1],
        "grade_j3"   : grades[2],
        **verdict,
    })

df = pd.DataFrame(records)
print(df["gold_status"].value_counts().to_string())
print(f"\n포함: {(df['gold_status']=='included').sum():,}  /  제외: {(df['gold_status']=='excluded').sum():,}")

gold_status
included    2519
excluded      24

포함: 2,519  /  제외: 24


## 5. 판정 결과 분포

In [17]:
print("=== confidence 분포 ===")
print(df["confidence"].value_counts().to_string())

print("\n=== final_grade 분포 (포함된 문서) ===")
included = df[df["gold_status"] == "included"]
print(included["final_grade"].value_counts().sort_index().to_string())

print("\n=== query_id별 포함/제외 ===")
summary = df.groupby("query_id")["gold_status"].value_counts().unstack(fill_value=0)
summary["total"] = summary.sum(axis=1)
print(summary.to_string())

# ── query_id별 grade 분포 상세 테이블 ──────────────────────────────────
# "excluded" = 3개 judge 합의 실패로 goldset에서 제외된 문서
#              (LLM API 실패와 무관 — judge 파일의 Failed 건수와 다름)
print("\n[query_id별 relevance_grade 분포]")

df["_label"] = df.apply(
    lambda r: f"grade {int(r['final_grade'])}" if r["gold_status"] == "included" else "excluded",
    axis=1,
)
pivot = df.groupby(["query_id", "_label"]).size().unstack(fill_value=0)
df.drop(columns=["_label"], inplace=True)

grade_cols = [f"grade {i}" for i in range(4) if f"grade {i}" in pivot.columns]
col_order  = grade_cols + (["excluded"] if "excluded" in pivot.columns else [])
pivot      = pivot.reindex(columns=col_order, fill_value=0)
pivot["total"] = pivot.sum(axis=1)

col_w = 9
qid_w = 10
header = f"  {'query_id':>{qid_w}}" + "".join(f"  {c:>{col_w}}" for c in col_order) + f"  {'total':>{col_w}}"
sep    = "-" * (len(header))
print(header)
print(sep)
for qid, row in pivot.iterrows():
    vals = "".join(f"  {int(row[c]):>{col_w}}" for c in col_order)
    print(f"  {qid:>{qid_w}}{vals}  {int(row['total']):>{col_w}}")

=== confidence 분포 ===
confidence
HIGH      1602
MEDIUM     917
LOW         24

=== final_grade 분포 (포함된 문서) ===
final_grade
0.0     241
1.0    1640
2.0     436
3.0     202

=== query_id별 포함/제외 ===
gold_status  excluded  included  total
query_id                              
S01                 2       113    115
S02                 0       115    115
S03                 0        85     85
S04                 0       102    102
S05                 0       130    130
S06                 1       119    120
S07                 0        67     67
S08                 0       125    125
S09                 1       136    137
S10                 2       114    116
S11                 0       141    141
S12                 3       122    125
S13                 1       136    137
S14                 1       129    130
S15                 0       114    114
S16                 1       144    145
S17                 2       114    116
S18                 1       121    122
S19                 5   

## 6. query_id별 grade 2~3 문서 충분성 점검

**최소 기준**: query_id별 grade ≥ 2 문서 3개 이상

In [18]:
MIN_GRADE  = 2
MIN_COUNT  = 3

high_grade = included[included["final_grade"] >= MIN_GRADE]
grade_count = high_grade.groupby("query_id").size().rename(f"grade>={MIN_GRADE}_count")

# 전체 query_id 기준으로 merge (없으면 0)
all_qids = pd.Series(sorted(df["query_id"].unique()), name="query_id")
check = all_qids.to_frame().merge(grade_count.reset_index(), on="query_id", how="left").fillna(0)
check[f"grade>={MIN_GRADE}_count"] = check[f"grade>={MIN_GRADE}_count"].astype(int)
check["sufficient"] = check[f"grade>={MIN_GRADE}_count"] >= MIN_COUNT

print(f"=== grade ≥ {MIN_GRADE} 문서 수 (최소 {MIN_COUNT}개 기준) ===")
print(check.to_string(index=False))

insufficient = check[~check["sufficient"]]
if insufficient.empty:
    print(f"\n✅ 모든 query_id가 최소 기준({MIN_COUNT}개) 충족")
else:
    print(f"\n⚠️  기준 미달 query_id ({len(insufficient)}개) → 재검토 필요:")
    print(insufficient.to_string(index=False))

=== grade ≥ 2 문서 수 (최소 3개 기준) ===
query_id  grade>=2_count  sufficient
     S01              18        True
     S02              10        True
     S03               5        True
     S04              18        True
     S05              14        True
     S06              64        True
     S07               9        True
     S08              19        True
     S09              20        True
     S10              50        True
     S11               8        True
     S12              61        True
     S13              31        True
     S14              16        True
     S15               5        True
     S16              22        True
     S17              41        True
     S18              76        True
     S19              76        True
     S20              36        True
     S21              39        True

✅ 모든 query_id가 최소 기준(3개) 충족


## 7. 제외된 문서 상세 (LOW 판정)

In [19]:
excluded = df[df["gold_status"] == "excluded"].copy()
excluded["grades"] = excluded.apply(lambda r: [r.grade_j1, r.grade_j2, r.grade_j3], axis=1)

print(f"제외 문서 총 {len(excluded)}건")
print("\n=== query_id별 제외 수 ===")
print(excluded["query_id"].value_counts().sort_index().to_string())

print("\n=== 제외 문서 샘플 (상위 10개) ===")
pd.set_option("display.max_colwidth", 25)
excluded[["query_id", "isbn", "title", "grades"]]

제외 문서 총 24건

=== query_id별 제외 수 ===
query_id
S01    2
S06    1
S09    1
S10    2
S12    3
S13    1
S14    1
S16    1
S17    2
S18    1
S19    5
S20    3
S21    1

=== 제외 문서 샘플 (상위 10개) ===


,query_id,isbn,title,grades
156,S10,9788932903316,생명있는 것은 다 사랑을 원한다,"[1, 2, 3]"
172,S12,9788933810460,구절리 바람소리 - 사 시인선 46,"[3, 1, 2]"
197,S19,9788935213733,다산의 마지막 질문 - 나를 깨닫는다는 것,"[3, 2, 1]"
198,S16,9788935659456,행복어사전 2 - 주 전집 24,"[3, 2, 1]"
267,S17,9788939230927,부어스 - 별을 따는 사람들,"[1, 2, 3]"
699,S19,9788967820602,아름다운 배웅 - 국내 첫 여성 장례지...,"[2, 1, 3]"
910,S12,9788979445213,꽃벼랑 - 김일연 시집,"[1, 2, 3]"
1161,S20,9788990024855,매드 사이언스 북 - 엉뚱하고 기발한 ...,"[3, 1, 2]"
1193,S17,9788990794963,문제아 - 년문학 보물창고 11,"[3, 1, 2]"
1312,S19,9788995050767,세상을 보는 지혜 - 전편 = Hand...,"[3, 1, 2]"


## 8. 최종 Goldset 저장

In [20]:
# 포함된 문서에 원본 메타데이터 결합
included_keys = set(zip(included["isbn"], included["query_id"]))

final_records = []
for key in sorted(included_keys):
    isbn, query_id = key
    base  = meta[key].copy()
    row   = df[(df["isbn"] == isbn) & (df["query_id"] == query_id)].iloc[0]

    base["final_grade"] = int(row["final_grade"])
    base["confidence"]  = row["confidence"]
    base["grade_j1"]    = int(row["grade_j1"])
    base["grade_j2"]    = int(row["grade_j2"])
    base["grade_j3"]    = int(row["grade_j3"])
    final_records.append(base)

out_path = DATA_DIR / "goldset_final.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(final_records, f, ensure_ascii=False, indent=2)

# out_jsonl = DATA_DIR / "goldset_final.jsonl"
# with open(out_jsonl, "w", encoding="utf-8") as f:
#     for r in final_records:
#         f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"저장 완료: {out_path}  ({len(final_records):,}건)")
# print(f"저장 완료: {out_jsonl}")

저장 완료: ../dataset/goldset_final.json  (2,519건)


## 9. 최종 Goldset 검증

In [21]:
final_df = pd.DataFrame(final_records)

print("=== 최종 Goldset 요약 ===")
print(f"총 문서 수    : {len(final_df):,}")
print(f"query_id 수   : {final_df['query_id'].nunique()}")

print("\n=== query_id별 grade 분포 ===")
pivot = final_df.groupby(["query_id", "final_grade"]).size().unstack(fill_value=0)
pivot.columns = [f"grade_{c}" for c in pivot.columns]
pivot["total"] = pivot.sum(axis=1)
if "grade_2" in pivot.columns and "grade_3" in pivot.columns:
    pivot["grade_2+3"] = pivot["grade_2"] + pivot["grade_3"]
elif "grade_2" in pivot.columns:
    pivot["grade_2+3"] = pivot["grade_2"]
elif "grade_3" in pivot.columns:
    pivot["grade_2+3"] = pivot["grade_3"]
print(pivot.to_string())

print("\n=== confidence 분포 ===")
print(final_df["confidence"].value_counts().to_string())

=== 최종 Goldset 요약 ===
총 문서 수    : 2,519
query_id 수   : 21

=== query_id별 grade 분포 ===
          grade_0  grade_1  grade_2  grade_3  total  grade_2+3
query_id                                                      
S01             9       86       11        7    113         18
S02            10       95        8        2    115         10
S03            33       47        4        1     85          5
S04             5       79       14        4    102         18
S05            11      105       11        3    130         14
S06             1       54       55        9    119         64
S07            30       28        6        3     67          9
S08            18       88       17        2    125         19
S09            15      101       14        6    136         20
S10             2       62       39       11    114         50
S11            20      113        8        0    141          8
S12             9       52        9       52    122         61
S13            15       90      

In [24]:
REVIEW_FIELDS = [
    "query_id", "isbn", "title", "author", "publish_date", "page",
    "large_cate", "mid_cate", "small_cate",
    "book_intro", "book_index", "review_text", 
    "confidence", "grade_j1", "grade_j2", "grade_j3", "final_grade", "llm_raw_output"
]


def get_min_rank(item):
    ri = item.get("retrieval_info", [])
    return min((r.get("rank", 9999) for r in ri), default=None)


def build_review_entry(item):
    min_rank = get_min_rank(item)
    entry = {k: item.get(k) for k in REVIEW_FIELDS}
    entry["semantic_query"]     = (item.get("rag_query") or {}).get("semantic_query", "")
    entry["min_retrieval_rank"] = min_rank
    entry["corrected_grade"]    = None   # 검수자가 grade 수정 시 기입 (0~3)
    return entry


# 1순위만 저장: MEDIUM × grade 2~3
review_items = [
    build_review_entry(item)
    for item in final_records
    if item.get("confidence") == "MEDIUM" and item.get("final_grade", 0) >= 2
]

# 정렬: query_id → min_retrieval_rank
review_items.sort(key=lambda x: (
    x.get("query_id", ""),
    x.get("min_retrieval_rank") or 9999,
))

# 저장
out_path = DATA_DIR / "goldset_review.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(review_items, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {out_path}  ({len(review_items):,}건)")
print(f"  MEDIUM × grade 2~3  (1순위 검수 대상)")
print(f"\n  corrected_grade 필드: null → 검수 후 수정할 grade(0~3) 기입")

저장 완료: ../dataset/goldset_review.json  (320건)
  MEDIUM × grade 2~3  (1순위 검수 대상)

  corrected_grade 필드: null → 검수 후 수정할 grade(0~3) 기입


In [ ]:
import math

mid = math.ceil(len(review_items) / 2)   # 320 → 160 / 160
half1 = review_items[:mid]
half2 = review_items[mid:]

def to_csv_df(items):
    return pd.DataFrame([
        {"isbn": r["isbn"], "grade": r["final_grade"], "corrected_grade": r["corrected_grade"]}
        for r in items
    ])

csv1 = DATA_DIR / "goldset_review_1.csv"
csv2 = DATA_DIR / "goldset_review_2.csv"
to_csv_df(half1).to_csv(csv1, index=False, encoding="utf-8-sig")
to_csv_df(half2).to_csv(csv2, index=False, encoding="utf-8-sig")

print(f"저장 완료")
print(f"  {csv1.name}  ({len(half1)}건)  query_id {half1[0]['query_id']} ~ {half1[-1]['query_id']}")
print(f"  {csv2.name}  ({len(half2)}건)  query_id {half2[0]['query_id']} ~ {half2[-1]['query_id']}")

## 11. 검수 파일 생성

`goldset_review.json` — 우선순위별 검수 대상을 하나의 파일로 통합

- `review_priority`: 1 / 2 / 3
- `min_retrieval_rank`: 검색 소스 중 가장 높은 rank (낮을수록 상위)
- `rank_thresholds`: 해당 항목이 포함되는 threshold 목록 (예: `[5, 10, 20]`)
- `semantic_query`: rag_query에서 추출한 검색 의도
- 2순위는 `min_rank ≤ 20` 이하 전체 포함 (threshold는 `rank_thresholds`로 필터링)

In [ ]:
def get_min_rank(item):
    ri = item.get("retrieval_info", [])
    return min((r.get("rank", 9999) for r in ri), default=None)


# 1순위: MEDIUM × grade 2~3
p1 = [d for d in final_records if d.get("confidence") == "MEDIUM" and d.get("final_grade", 0) >= 2]

# 2순위: grade 1 전체 (threshold별 필터링)
grade1_all = [d for d in final_records if d.get("final_grade") == 1]

# 3순위: LOW (excluded df는 위 셀에서 정의됨)
p3_cnt = len(excluded)

print("=== 검수 우선순위 갯수 ===")
print(f"1순위  MEDIUM × grade 2~3         : {len(p1):>5}건")
print()
for t in [5, 10, 20]:
    n = sum(1 for d in grade1_all if get_min_rank(d) is not None and get_min_rank(d) <= t)
    print(f"2순위  grade 1 × min_rank ≤ {t:2d}   : {n:>5}건")
print()
print(f"3순위  LOW (합의 실패, excluded)   : {p3_cnt:>5}건")

## 10. 검수 우선순위 갯수 확인

| 우선순위 | 조건 | 설명 |
|----------|------|------|
| 1순위 | MEDIUM × grade 2~3 | judge 2/3 합의, 사람 검수 필요 |
| 2순위 | grade 1 × retrieval 상위권 | Hard Negative 후보 (threshold 결정 필요) |
| 3순위 | LOW (합의 실패) | judge 3명 모두 다른 grade |